<!--nav--> [🗺 Learning path](README.md) · **1/37** · [Simple MultiGPU ActualTraining](./Simple_MultiGPU_ActualTraining.ipynb) ▶

# Simple Multi-GPU Training

The simplest possible distributed fine-tuning notebook.

- **Model:** GPT-2 (124M params) — fits easily on free GPUs
- **Method:** LoRA + DeepSpeed ZeRO-2 + Accelerate
- **Data:** 500 examples from Alpaca
- **Platform:** Kaggle 2x T4 (free) or Colab 1x T4 (free)

No SFT/DPO trainers — just HuggingFace `Trainer` with a causal LM.

In [ ]:
!pip install -q transformers datasets peft accelerate deepspeed

In [ ]:
import torch, os, json, time

assert torch.cuda.is_available(), "GPU required!"

NUM_GPUS = torch.cuda.device_count()
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.0f} GB)")
print(f"\nTotal GPUs: {NUM_GPUS}")

<cell_type>markdown</cell_type>## Write Config

Accelerate config for DeepSpeed ZeRO-2.

In [ ]:
# Accelerate config — use accelerate-managed DeepSpeed (no deepspeed_config_file)
# This avoids the conflict where accelerate rejects overlapping settings
accel_yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  gradient_accumulation_steps: auto
  gradient_clipping: auto
  offload_optimizer_device: cpu
  offload_param_device: none
  zero3_init_flag: false
  zero_stage: 2
machine_rank: 0
main_process_ip: null
main_process_port: null
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: {NUM_GPUS}
use_cpu: false
"""
accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
with open(os.path.join(accel_dir, "default_config.yaml"), "w") as f:
    f.write(accel_yaml)

print(f"Config written. {NUM_GPUS} GPU(s), ZeRO-2, bf16.")

## Write Training Script

Plain HuggingFace `Trainer` — no TRL needed. Just causal language modeling with LoRA.

In [ ]:
%%writefile train.py
"""Simple distributed fine-tuning: GPT-2 + LoRA + DeepSpeed."""
import torch, os, json, time
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model

MODEL = "gpt2"  # 124M params — tiny and fast

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16)
total_params = sum(p.numel() for p in model.parameters())

# Add LoRA
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["c_attn"],  # GPT-2 attention
    bias="none", task_type="CAUSAL_LM",
))
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model.print_trainable_parameters()

# Load data — 500 Alpaca examples, formatted as plain text
dataset = load_dataset("tatsu-lab/alpaca", split="train")
dataset = dataset.shuffle(seed=42).select(range(500))

def format_and_tokenize(ex):
    text = f"### Instruction:\n{ex['instruction']}"
    if ex.get("input"):
        text += f"\n### Input:\n{ex['input']}"
    text += f"\n### Response:\n{ex['output']}{tokenizer.eos_token}"
    return tokenizer(text, truncation=True, max_length=512, padding=False)

dataset = dataset.map(format_and_tokenize, remove_columns=dataset.column_names)
print(f"Dataset: {len(dataset)} examples")

# Callback to collect detailed metrics
class MetricsCallback(TrainerCallback):
    def __init__(self):
        self.logs = []
        self.step_times = []
        self.train_start = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.time()
        torch.cuda.reset_peak_memory_stats()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            entry = {
                "step": state.global_step,
                "loss": logs["loss"],
                "learning_rate": logs.get("learning_rate", 0),
                "epoch": logs.get("epoch", 0),
            }
            self.logs.append(entry)

    def on_step_end(self, args, state, control, **kwargs):
        self.step_times.append(time.time())

    def on_train_end(self, args, state, control, **kwargs):
        if int(os.environ.get("LOCAL_RANK", 0)) != 0:
            return
        train_time = time.time() - self.train_start

        # GPU memory stats
        gpu_mem_allocated = torch.cuda.max_memory_allocated() / 1e9
        gpu_mem_reserved = torch.cuda.max_memory_reserved() / 1e9

        # Throughput
        num_gpus = int(os.environ.get("WORLD_SIZE", 1))
        total_steps = state.global_step
        total_samples = state.global_step * args.per_device_train_batch_size * args.gradient_accumulation_steps * num_gpus

        # Estimate tokens processed (avg tokens per example from dataset)
        avg_tokens = 120  # rough avg for Alpaca examples
        total_tokens = total_samples * avg_tokens

        # Step throughput
        if len(self.step_times) > 2:
            step_durations = [self.step_times[i+1] - self.step_times[i] for i in range(len(self.step_times)-1)]
            avg_step_time = sum(step_durations) / len(step_durations)
        else:
            avg_step_time = train_time / max(total_steps, 1)

        samples_per_sec = total_samples / train_time
        tokens_per_sec = total_tokens / train_time

        metrics = {
            "model": MODEL,
            "total_params": total_params,
            "trainable_params": trainable_params,
            "trainable_pct": trainable_params / total_params * 100,
            "num_gpus": num_gpus,
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_mem_allocated_gb": round(gpu_mem_allocated, 2),
            "gpu_mem_reserved_gb": round(gpu_mem_reserved, 2),
            "gpu_mem_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
            "train_time_sec": round(train_time, 1),
            "total_steps": total_steps,
            "total_samples": total_samples,
            "total_tokens_approx": total_tokens,
            "samples_per_sec": round(samples_per_sec, 2),
            "tokens_per_sec": round(tokens_per_sec, 1),
            "avg_step_time_sec": round(avg_step_time, 3),
            "final_loss": self.logs[-1]["loss"] if self.logs else None,
            "loss_history": self.logs,
            "batch_size_per_gpu": args.per_device_train_batch_size,
            "grad_accum_steps": args.gradient_accumulation_steps,
            "effective_batch_size": args.per_device_train_batch_size * args.gradient_accumulation_steps * num_gpus,
            "learning_rate": args.learning_rate,
        }
        with open("training_metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"\nMetrics saved to training_metrics.json")

metrics_cb = MetricsCallback()

# Train — DataCollatorForLanguageModeling handles labels + padding automatically
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./output",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=1,
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        save_strategy="no",
    ),
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[metrics_cb],
)

trainer.train()
trainer.save_model("./output/final")
tokenizer.save_pretrained("./output/final")

# Quick generation test (rank 0 only)
if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    model.eval()
    prompts = [
        "### Instruction:\nWhat is gravity?\n### Response:\n",
        "### Instruction:\nWrite a haiku about coding.\n### Response:\n",
    ]
    print("\n" + "=" * 50)
    for p in prompts:
        inputs = tokenizer(p, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=80, temperature=0.7, do_sample=True)
        response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"Q: {p.split(chr(10))[1].replace('### Instruction:', '').strip()}")
        print(f"A: {response.strip()[:200]}\n")
    print("=" * 50)

## Launch Training

In [ ]:
print(f"Launching on {NUM_GPUS} GPU(s)...")
start = time.time()

!accelerate launch --num_processes={NUM_GPUS} train.py

elapsed = time.time() - start
print(f"\nDone! {elapsed:.0f}s on {NUM_GPUS}x {torch.cuda.get_device_name(0)}")

## Performance Dashboard

In [ ]:
import json, matplotlib.pyplot as plt, matplotlib.ticker as ticker
from IPython.display import HTML, display
import numpy as np

with open("training_metrics.json") as f:
    m = json.load(f)

# --- Loss Curve + Learning Rate (dual axis) ---
fig, ax1 = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor("#0d1117")
ax1.set_facecolor("#0d1117")

steps = [h["step"] for h in m["loss_history"]]
losses = [h["loss"] for h in m["loss_history"]]
lrs = [h["learning_rate"] for h in m["loss_history"]]

# Loss
color_loss = "#58a6ff"
ax1.plot(steps, losses, color=color_loss, linewidth=2, label="Loss", zorder=3)
ax1.fill_between(steps, losses, alpha=0.1, color=color_loss)
ax1.set_xlabel("Step", color="#8b949e", fontsize=11)
ax1.set_ylabel("Loss", color=color_loss, fontsize=11)
ax1.tick_params(axis="y", labelcolor=color_loss)
ax1.tick_params(axis="x", colors="#8b949e")
ax1.grid(True, alpha=0.15, color="#30363d")
ax1.spines["top"].set_visible(False)
for spine in ax1.spines.values():
    spine.set_color("#30363d")

# Learning rate on right axis
ax2 = ax1.twinx()
color_lr = "#f0883e"
ax2.plot(steps, lrs, color=color_lr, linewidth=1.5, linestyle="--", alpha=0.7, label="LR")
ax2.set_ylabel("Learning Rate", color=color_lr, fontsize=11)
ax2.tick_params(axis="y", labelcolor=color_lr)
ax2.spines["top"].set_visible(False)
for spine in ax2.spines.values():
    spine.set_color("#30363d")
ax2.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1e"))

fig.suptitle("Training Loss & Learning Rate", color="#e6edf3", fontsize=14, fontweight="bold")
fig.legend(loc="upper right", bbox_to_anchor=(0.92, 0.88), facecolor="#161b22", edgecolor="#30363d",
           labelcolor="#e6edf3", fontsize=10)
plt.tight_layout()
plt.show()

# --- GPU Memory Bar Chart ---
fig2, ax3 = plt.subplots(figsize=(6, 3))
fig2.patch.set_facecolor("#0d1117")
ax3.set_facecolor("#0d1117")

mem_labels = ["Allocated", "Reserved", "Total"]
mem_vals = [m["gpu_mem_allocated_gb"], m["gpu_mem_reserved_gb"], m["gpu_mem_total_gb"]]
colors = ["#3fb950", "#58a6ff", "#30363d"]
bars = ax3.barh(mem_labels, mem_vals, color=colors, height=0.5, edgecolor="#0d1117")
for bar, val in zip(bars, mem_vals):
    ax3.text(val + 0.1, bar.get_y() + bar.get_height()/2, f"{val:.1f} GB",
             va="center", color="#e6edf3", fontsize=11, fontweight="bold")
ax3.set_xlim(0, m["gpu_mem_total_gb"] * 1.3)
ax3.set_title(f"GPU Memory — {m['gpu_name']}", color="#e6edf3", fontsize=13, fontweight="bold")
ax3.tick_params(colors="#8b949e")
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
for spine in ax3.spines.values():
    spine.set_color("#30363d")
plt.tight_layout()
plt.show()

# --- HTML Summary Dashboard ---
loss_drop = ""
if len(losses) >= 2:
    pct = (losses[0] - losses[-1]) / losses[0] * 100
    loss_drop = f"{pct:.0f}% drop"

mem_util = m["gpu_mem_allocated_gb"] / m["gpu_mem_total_gb"] * 100

html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            max-width: 780px; margin: 20px 0;">

  <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 16px;">
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Throughput</div>
      <div style="color: #58a6ff; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['samples_per_sec']:.1f}</div>
      <div style="color: #8b949e; font-size: 12px;">samples/sec</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Token Speed</div>
      <div style="color: #3fb950; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['tokens_per_sec']:,.0f}</div>
      <div style="color: #8b949e; font-size: 12px;">tokens/sec</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Final Loss</div>
      <div style="color: #f0883e; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['final_loss']:.3f}</div>
      <div style="color: #8b949e; font-size: 12px;">{loss_drop}</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Train Time</div>
      <div style="color: #d2a8ff; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['train_time_sec']:.0f}s</div>
      <div style="color: #8b949e; font-size: 12px;">{m['total_steps']} steps</div>
    </div>
  </div>

  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px;">
    <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
      <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 12px;
                  border-bottom: 1px solid #21262d; padding-bottom: 8px;">Model & Training</div>
      <table style="width:100%; color: #c9d1d9; font-size: 12px; border-spacing: 0 6px;">
        <tr><td style="color:#8b949e;">Model</td><td style="text-align:right; font-weight:600;">{m['model']}</td></tr>
        <tr><td style="color:#8b949e;">Total params</td><td style="text-align:right;">{m['total_params']/1e6:.0f}M</td></tr>
        <tr><td style="color:#8b949e;">Trainable (LoRA)</td>
            <td style="text-align:right; color:#3fb950;">{m['trainable_params']/1e3:.0f}K ({m['trainable_pct']:.2f}%)</td></tr>
        <tr><td style="color:#8b949e;">Batch size (effective)</td>
            <td style="text-align:right;">{m['batch_size_per_gpu']} x {m['grad_accum_steps']} x {m['num_gpus']}GPU = {m['effective_batch_size']}</td></tr>
        <tr><td style="color:#8b949e;">Learning rate</td><td style="text-align:right;">{m['learning_rate']}</td></tr>
        <tr><td style="color:#8b949e;">Precision</td><td style="text-align:right;">bf16</td></tr>
      </table>
    </div>
    <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
      <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 12px;
                  border-bottom: 1px solid #21262d; padding-bottom: 8px;">GPU & Memory</div>
      <table style="width:100%; color: #c9d1d9; font-size: 12px; border-spacing: 0 6px;">
        <tr><td style="color:#8b949e;">GPUs</td><td style="text-align:right; font-weight:600;">{m['num_gpus']}x {m['gpu_name']}</td></tr>
        <tr><td style="color:#8b949e;">VRAM total</td><td style="text-align:right;">{m['gpu_mem_total_gb']} GB per GPU</td></tr>
        <tr><td style="color:#8b949e;">Peak allocated</td>
            <td style="text-align:right; color:#3fb950;">{m['gpu_mem_allocated_gb']} GB ({mem_util:.0f}%)</td></tr>
        <tr><td style="color:#8b949e;">Peak reserved</td><td style="text-align:right;">{m['gpu_mem_reserved_gb']} GB</td></tr>
        <tr><td style="color:#8b949e;">Optimizer</td><td style="text-align:right;">CPU offload (ZeRO-2)</td></tr>
        <tr><td style="color:#8b949e;">Avg step time</td><td style="text-align:right;">{m['avg_step_time_sec']*1000:.0f} ms</td></tr>
      </table>
    </div>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 14px 18px;
              margin-top: 12px; display: flex; justify-content: space-between; align-items: center;">
    <span style="color: #8b949e; font-size: 12px;">GPU Memory Utilization</span>
    <div style="flex: 1; margin: 0 16px; background: #21262d; border-radius: 6px; height: 18px; overflow: hidden;">
      <div style="width: {mem_util:.0f}%; height: 100%; border-radius: 6px;
                  background: linear-gradient(90deg, #238636, #3fb950);"></div>
    </div>
    <span style="color: #3fb950; font-size: 13px; font-weight: 700;">{mem_util:.0f}%</span>
  </div>

</div>
"""
display(HTML(html))

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
import torch

model = AutoPeftModelForCausalLM.from_pretrained("./output/final", dtype=torch.bfloat16).to("cuda")
tokenizer = AutoTokenizer.from_pretrained("./output/final")
model.eval()

questions = ["Explain recursion simply.", "What makes a good leader?", "How do computers store data?"]
for q in questions:
    prompt = f"### Instruction:\n{q}\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    print(f"Q: {q}")
    print(f"A: {tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()[:200]}\n")

---

## How It Works

```
accelerate launch --num_processes=N train.py
        |
   Worker 0 (GPU 0)    Worker 1 (GPU 1)
        |                    |
   DeepSpeed ZeRO-2: splits optimizer across GPUs + CPU
        |                    |
   LoRA: trains <1% of params (tiny memory footprint)
```

The training script is **identical** to single-GPU code. The only distributed parts:
1. The accelerate config YAML (written once, sets ZeRO-2 + CPU offload)
2. `accelerate launch` to run it

| Platform | GPUs | Cost |
|----------|------|------|
| **Kaggle** | 2x T4 | Free (30h/week) |
| **Colab** | 1x T4 | Free |